# MNE-bids

In [ ]:
%matplotlib qt

In [ ]:
from pathlib import Path
import numpy as np

import openneuro
import mne
from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
)

import matplotlib.pyplot as plt
plt.switch_backend('Qt5Agg')
plt.ion()

In [ ]:
dataset = "ds003969"
tasks = ['think1', 'think2','med1breath', 'med2']

# Create a BIDS root directory in the MNE dataset folder
root = Path('/home/jackhoyle')
bids_root = root / 'mne_data' / dataset
bids_root.mkdir(parents=True, exist_ok=True)

initially I've just downloaded subject 1 due to bandwidth constraints...

In [ ]:
subject = "001"
task = 'med2'

In [ ]:
raw: mne.io.edf.edf.RawEDF = read_raw_bids(
    BIDSPath(subject=subject, root=bids_root, task=task, datatype="eeg"),
    verbose=True
)
raw.load_data()
assert type(raw) == mne.io.edf.edf.RawEDF

# data processing

In [ ]:
#1: set reference
raw.set_eeg_reference(['EXG6']) # EXG6 is right mastoid

In [ ]:
#2: subsample to 256 Hz
raw.resample(256)

In [ ]:
#3: apply a high pass filter at 1Hz
iir_params = dict(order=6, ftype='butter')
raw.filter(l_freq=1.0, h_freq=None, h_trans_bandwidth=0.3, method='iir', iir_params=iir_params)

In [ ]:
#4: "pop_rejcont" automated rejection of artifact segments (port of EEGLAB's pop_rejcont, see code/preprocessing.py)
# 1 s epochs, 0.5 s step (EEGLAB 'overlap'=0.5), 0-10 Hz > 17 dB and 35-128 Hz > 14 dB,
# contiguous=8 (-> merge distance 4 s), thresholds relative to the recording's mean spectrum.
import sys
sys.path.insert(0, str(Path.cwd().parent / 'code'))
from preprocessing import rejcont
regions = rejcont(raw, bands=((0, 10, 17), (35, 128, 14)), epochlength=1.0,
                  recurrence=0.5, contiguous=8, addlength=0.25, picks='eeg')
print(f"{len(regions)} regions, {sum(d for _, d in regions):.1f} s of {raw.times[-1]:.0f} s rejected")

In [ ]:
# 5. Visual inspection and potential adjustment of rejection thresholds?
#  "The data was then checked visually for
# any potential remaining artifact—leading in some rare cases to adjust decibels thresholds from
# 1 to 5 dB and rerunning the rejection procedure."

# for now I'm going to skip this as it's super unclear what they did

# 6. Removing bad electrodes
"We then manually identified and removed bad electrodes (from 0 to 18 bad electrodes per subject, average of 5 electrodes removed per subject)."

I think this needs to be done and is a good exercise in using the tools here.
In this notebook I'll do this for the test case I have, but I'll use a dedicated notebook to gradually
go through all the subjects and identify bad electrodes.

Something I've noticed: looking at channels after changing the reference can be misleading: Fp1 is bad (reading near zero voltage throughout), but when we reference it looks normal post the other processing

I have more leeway here as the authors don't specify how they found the bad channels. What makes the most sense to me here is loading the raw data, subsampling, applying the filters, doing rejcont so we can double check that, then inspecting the plots. Basically just omit referencing.


In [ ]:
# inspect and mark bad
raw.plot(duration=60, picks='eeg', block=True)

# inspection

In [ ]:
raw.plot(duration=30)

In [ ]:
(picks := [x for x in raw.ch_names if 'Fp' in x])

In [ ]:
raw.plot(duration=60, picks=picks)

In [ ]:
raw.compute_psd(picks=picks).plot();